# Roulette Analytics Lab V2

A reproducible research notebook for **casino roulette**. It imports the tested production package and presents named generated CSV outputs rather than maintaining notebook-only formulas.

In [1]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

from roulette_lab.analysis import AnalysisConfig, run_full_analysis
from roulette_lab.bets import expected_net_return
from roulette_lab.decision import posterior_edge_summary
from roulette_lab.risk import build_risk_frontier
from roulette_lab.sequential import cusum_change_detection, likelihood_ratio_path
from roulette_lab.statistics import max_count_test

TABLES = ROOT / 'outputs' / 'tables'
tables = {path.stem: pd.read_csv(path) for path in TABLES.glob('*.csv')}
pd.options.display.float_format = '{:.6f}'.format
production_interfaces = (run_full_analysis, expected_net_return, max_count_test, likelihood_ratio_path, cusum_change_detection, posterior_edge_summary, build_risk_frontier)
print(f'Loaded {len(tables)} generated CSV tables and {len(production_interfaces)} production interfaces.')

Loaded 11 generated CSV tables and 7 production interfaces.


## Exact Wheel Economics

The casino contract comes first. `house_edges.csv` reports exact expected net returns and house edges for European and American wheels, plus the two implemented European special rules. A fair wheel is not the same as a fair wager.

In [2]:
# Generated evidence: outputs/tables/house_edges.csv
tables['house_edges'][['rule', 'bet', 'expected_net_return', 'house_edge']]

,rule,bet,expected_net_return,house_edge
0,european,straight,-0.027027,0.027027
1,american,straight,-0.052632,0.052632
2,european_standard,red,-0.027027,0.027027
3,la_partage,red,-0.013514,0.013514
4,en_prison,red,-0.013514,0.013514


## Random Walk and Law of Large Numbers

The cumulative frequency table makes the convergence claim concrete. A finite random walk can rise while the fair-wheel expected return remains negative; long-run frequency convergence is not a promise about a single path.

In [3]:
# Generated evidence: outputs/tables/lln_convergence.csv
tables['lln_convergence'].iloc[[0, 9, 99, -1]][['spin', 'empirical_probability', 'theoretical_probability', 'absolute_error']]

,spin,empirical_probability,theoretical_probability,absolute_error
0,1,0.000000,0.027027,0.027027
9,10,0.000000,0.027027,0.027027
99,100,0.020000,0.027027,0.007027
19999,20000,0.026900,0.027027,0.000127


## Fixed-Horizon Fairness

A global chi-square test examines the complete pre-defined count vector. The Monte Carlo column provides a multinomial calibration for the same fixed-horizon question.

In [4]:
# Generated evidence: outputs/tables/bias_tests.csv
tables['bias_tests'][['dataset', 'spins', 'chi_square_statistic', 'asymptotic_p_value', 'monte_carlo_global_p_value']]

,dataset,spins,chi_square_statistic,asymptotic_p_value,monte_carlo_global_p_value
0,unbiased,1000,37.554000,0.397832,0.397360
1,biased,1000,63.824000,0.002893,0.004400


## Selection Correction

The hottest pocket was chosen after inspecting all pockets. `bias_tests.csv` therefore keeps the naive one-pocket tail beside Bonferroni and maximum-count family-wise values. The latter represents the actual search procedure.

In [5]:
# Generated evidence: outputs/tables/bias_tests.csv
tables['bias_tests'][['dataset', 'hottest_label', 'observed_max', 'naive_p_value', 'bonferroni_p_value', 'familywise_p_value']]

,dataset,hottest_label,observed_max,naive_p_value,bonferroni_p_value,familywise_p_value
0,unbiased,32,39,0.016397,0.606686,0.483652
1,biased,17,61,0.000000,0.000000,0.000100


## Sequential Evidence

Repeated fixed-horizon peeking changes its calibration. The production likelihood-ratio process uses a pre-specified simple null, simple alternative, and threshold. The displayed table is generated evidence, not a notebook calculation.

In [6]:
# Generated evidence: outputs/tables/sequential_evidence.csv
tables['sequential_evidence'].groupby('scenario', as_index=False).tail(1)[['scenario', 'spin', 'cumulative_hits', 'e_value', 'e_value_threshold', 'first_crossing', 'fixed_horizon_p_value']]

,scenario,spin,cumulative_hits,e_value,e_value_threshold,first_crossing,fixed_horizon_p_value
999,fair_null,1000,26,0.000003,20,0,0.606177


## CUSUM Change-Point Diagnostic

CUSUM is a targeted diagnostic for a declared upward change. It reports alarms and simulated delay for the synthetic scenario, but it cannot establish the physical cause or exact timing of a real change.

In [7]:
# Generated evidence: outputs/tables/change_point_results.csv
tables['change_point_results'].groupby('scenario', as_index=False).tail(1)[['scenario', 'cusum_threshold', 'first_alarm', 'detection_delay', 'false_alarm_rate', 'median_detection_delay']]

,scenario,cusum_threshold,first_alarm,detection_delay,false_alarm_rate,median_detection_delay
999,fair_null,4,0,-1,0.149667,-1
1999,changed_at_500,4,910,410,0.070333,153


## Posterior Edge Uncertainty

The Beta posterior summary separates a point estimate from uncertainty. The target is a labelled synthetic teaching case; it does not establish a live casino probability.

In [8]:
# Generated evidence: outputs/tables/posterior_edge.csv
tables['posterior_edge'][['dataset', 'label', 'posterior_mean', 'credible_interval_lower', 'credible_interval_upper', 'break_even_probability', 'probability_positive_edge']]

,dataset,label,posterior_mean,credible_interval_lower,credible_interval_upper,break_even_probability,probability_positive_edge
0,biased_teaching_sample,17,0.059788,0.046188,0.074995,0.027778,1.000000


## Robust Kelly Decisions

Kelly is constrained expected-log-growth optimisation under an assumed edge, odds, and repeated-trial model. Fair roulette has zero Kelly. The lower posterior-quantile version is explicitly a conservative heuristic, not a guaranteed or universal optimum.

In [9]:
# Generated evidence: outputs/tables/posterior_edge.csv
tables['posterior_edge'][['plugin_kelly', 'quantile_probability', 'quantile_kelly', 'quantile_kelly_interpretation']]

,plugin_kelly,quantile_probability,quantile_kelly,quantile_kelly_interpretation
0,0.032925,0.050554,0.023427,heuristic_lower_posterior_quantile


## CVaR Risk Frontier

The frontier uses common random numbers to compare stake fractions. CVaR is the mean of the worst non-negative terminal shortfalls `max(initial bankroll - terminal equity, 0)` at the recorded tail probability, so larger values are worse.

In [10]:
# Generated evidence: outputs/tables/risk_frontier.csv
tables['risk_frontier'][['kelly_fraction_multiplier', 'expected_log_growth', 'probability_of_loss', 'expected_maximum_drawdown', 'terminal_cvar_shortfall', 'cvar_tail_probability']]

,kelly_fraction_multiplier,expected_log_growth,probability_of_loss,expected_maximum_drawdown,terminal_cvar_shortfall,cvar_tail_probability
0,0.250000,0.751662,0.020667,0.242072,148.286667,0.050000
1,0.500000,0.640320,0.130667,0.323899,505.160000,0.050000
2,1.000000,0.366748,0.370667,0.386158,510.093333,0.050000


## Limitations

The simulations are conditional on named synthetic scenarios, probabilities, seed, horizon, and table constraints. Fixed-horizon, post-selection, sequential, and change-point questions need different procedures. None of these outputs is gambling advice or a claim about a live wheel.